In [53]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [54]:
import numpy as np
from pygrad.tensor.tensor import Tensor
from pygrad.optimizers.sgd import SGD
from pygrad.optimizers.backprop import backward
from ucimlrepo import fetch_ucirepo 

In [55]:
np.random.RandomState = np.random.mtrand.RandomState
np.random.seed(52)

In [56]:
# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets 
  
# metadata 
print(wine_quality.metadata) 
  
# variable information 
print(wine_quality.variables) 

{'uci_id': 186, 'name': 'Wine Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/186/wine+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/186/data.csv', 'abstract': 'Two datasets are included, related to red and white vinho verde wine samples, from the north of Portugal. The goal is to model wine quality based on physicochemical tests (see [Cortez et al., 2009], http://www3.dsi.uminho.pt/pcortez/wine/).', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 4898, 'num_features': 11, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['quality'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Wed Nov 15 2023', 'dataset_doi': '10.24432/C56S3T', 'creators': ['Paulo Cortez', 'A. Cerdeira', 'F. Almeida', 'T. Matos', 'J. Reis'], 'intro_paper': {'ID': 252, 'type': 'NATIVE', 'title': 'Modeling wine preferences

In [57]:
X = X.to_numpy()
y = y.to_numpy()

In [58]:
X[0]

array([ 7.4   ,  0.7   ,  0.    ,  1.9   ,  0.076 , 11.    , 34.    ,
        0.9978,  3.51  ,  0.56  ,  9.4   ])

In [59]:
y = y.squeeze(axis=1)
y = y - 3
y

array([2, 2, 2, ..., 3, 4, 3], shape=(6497,))

In [60]:
y = np.eye(7)[y]
y

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(6497, 7))

In [61]:
counts = y.sum(axis=0)
weights = len(y) / (7 * counts)

In [62]:
weights

array([ 30.93809524,   4.29695767,   0.43411733,   0.32727181,
         0.860188  ,   4.80903035, 185.62857143])

In [63]:
test_counts = np.maximum(1, np.rint(350 * counts / len(y)).astype(int))
test_counts[np.argmax(test_counts)] -= test_counts.sum() - 350

In [64]:
test_indx = np.concatenate([
    np.random.permutation(np.flatnonzero(y[:, class_id]))[:n]
    for class_id, n in enumerate(test_counts)
])
train_indx = np.setdiff1d(np.arange(len(X)), test_indx)
np.random.shuffle(test_indx)
np.random.shuffle(train_indx)

In [65]:
X_train = X[train_indx]
y_train = y[train_indx]

X_test = X[test_indx]
y_test = y[test_indx]

In [66]:
mean_x = np.mean(X_train, axis=0)
std_x = np.std(X_train, axis=0)
X_train_scaled = (X_train - mean_x) / std_x
X_test_scaled = (X_test - mean_x) / std_x

In [67]:
model_width = 42

In [68]:
W_1 = Tensor(
    np.random.randn(len(X[0]), model_width)*0.1,
    device='cuda',
    requires_grad=True
)
W_1

Tensor([[ 8.09400752e-02  1.82442904e-01  1.07373588e-01 -1.42091095e-01
   3.02981399e-03 -3.20124552e-02 -1.07149087e-01  9.15050581e-02
   3.01240459e-02 -2.22973943e-01 -2.02583492e-01 -8.63398388e-02
  -6.60278276e-02 -1.49740711e-01 -1.94267780e-02 -8.94043520e-02
  -5.89230098e-03  1.93771515e-02  1.76147148e-01  2.37515997e-02
  -1.15229741e-01 -7.89790321e-03 -2.62452941e-02 -4.17580269e-02
  -2.46422708e-01  1.48236245e-01 -8.58177915e-02  1.75014585e-02
  -9.52149928e-02 -1.42764702e-01 -1.06139584e-02  6.48454353e-02
  -2.56335028e-02 -6.03073724e-02 -8.76019895e-03  8.12330842e-02
  -1.30844414e-01  7.69946277e-02 -7.21204653e-02  5.26432283e-02
   2.72341698e-01  3.41976318e-03]
 [-1.80507332e-01 -9.22505707e-02 -6.78866059e-02 -3.29058468e-02
  -1.59110487e-01 -1.67132586e-01  2.40591049e-01  3.76064442e-02
   1.86860599e-02  7.21993446e-02 -4.81414832e-02 -3.91278602e-02
   1.06146231e-01  7.95240179e-02  4.90383357e-02  8.66418257e-02
  -8.57926607e-02 -1.21210173e-01 

In [69]:
W_2 = Tensor(
    np.random.randn(model_width, model_width)*0.1,
    device='cuda',
    requires_grad=True
)
W_2

Tensor([[-0.1192278   0.2369179  -0.1907919  ... -0.00105612 -0.12611926
  -0.02847156]
 [ 0.12805735 -0.04911979  0.0174996  ... -0.09778665  0.00059882
  -0.08078877]
 [-0.10281154 -0.06554882  0.12356751 ...  0.15588886 -0.08069735
  -0.08213603]
 ...
 [ 0.06405429 -0.01705283 -0.05021537 ...  0.03055683  0.13155818
  -0.02477722]
 [-0.09453616  0.1440221   0.0103206  ... -0.06618799 -0.1531349
   0.18977271]
 [-0.1563329   0.07299428 -0.06665231 ...  0.10365283 -0.01333168
   0.1622396 ]]) (0x7f87ddd40cb0))

In [70]:
W_3 = Tensor(
    np.random.randn(model_width, model_width)*0.1,
    device='cuda',
    requires_grad=True
)
W_3

Tensor([[-0.0429897  -0.0179726  -0.02120967 ... -0.09707987 -0.00610074
   0.1084904 ]
 [-0.12375473  0.05234373 -0.06583814 ...  0.07279424 -0.02228467
   0.00580762]
 [-0.15467948  0.06830269  0.02078006 ... -0.17299041  0.18763837
   0.10891531]
 ...
 [-0.01270144  0.06411562  0.27223593 ... -0.06018247 -0.12402346
  -0.03609509]
 [ 0.34641784  0.01386986  0.01273796 ... -0.07362249 -0.05877214
   0.16346195]
 [-0.04417815 -0.05081511 -0.25393724 ... -0.21664113  0.20086142
  -0.2211769 ]]) (0x7f87ddd02d80))

In [71]:
W_4 = Tensor(
    np.random.randn(model_width, 7)*0.1,
    device='cuda',
    requires_grad=True
)
W_4

Tensor([[ 0.09743172  0.05559473 -0.04716255  0.02048663  0.0234348   0.00656358
   0.1184935 ]
 [-0.10040324  0.06684479  0.11675548 -0.0531088   0.01572039  0.14075716
   0.03512386]
 [ 0.02485619  0.15908602  0.06067844 -0.00497625 -0.1340172   0.04157151
   0.02488477]
 [-0.09862109  0.19755477  0.1005295   0.22881532  0.00911842  0.06919044
   0.00066351]
 [-0.04870003 -0.21878465  0.18920827 -0.01767926  0.15942949 -0.19620259
   0.03713237]
 [-0.11115677  0.14481938 -0.02340872 -0.20821361 -0.02032683 -0.10007397
   0.00271414]
 [ 0.07478339 -0.00870365  0.19396448 -0.21624228  0.11699005  0.01676078
  -0.12697966]
 [ 0.10177378 -0.20597786  0.01749643 -0.02166048 -0.2072697  -0.0146874
  -0.12731911]
 [ 0.0215877  -0.05465781  0.0095235  -0.113051    0.03185595  0.2174162
   0.05293676]
 [-0.06080059  0.02253205 -0.10003429  0.09326145 -0.08955956  0.0538116
   0.14298   ]
 [ 0.00380908  0.10481634 -0.00866713 -0.0562838   0.00713222  0.07520518
  -0.14528416]
 [ 0.0023448  -0.

In [72]:
b_1 = Tensor(np.zeros((1, model_width)), 'cuda', requires_grad=True)
b_1

Tensor([[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]) (0x7f87ddd41c10))

In [73]:
b_2 = Tensor(np.zeros((1, model_width)), 'cuda', requires_grad=True)
b_2

Tensor([[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]) (0x7f87ddde58b0))

In [74]:
b_3 = Tensor(np.zeros((1, model_width)), 'cuda', requires_grad=True)
b_3

Tensor([[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]) (0x7f87ddd40350))

In [75]:
b_4 = Tensor(np.zeros((1, 7)), 'cuda', requires_grad=True)
b_4

Tensor([[0. 0. 0. 0. 0. 0. 0.]]) (0x7f87de12bc20))

In [76]:
def forward(X, W_1, W_2, W_3, W_4, b_1, b_2, b_3, b_4):
    h_1 = X@W_1 + b_1
    a_1 = h_1.relu()
    
    h_2 = a_1@W_2 + b_2
    a_2 = h_2.relu()
    
    h_3 = a_2@W_3 + b_3
    a_3 = h_3.relu()
    
    h_4 = a_3@W_4 + b_4
    
    return h_4

In [77]:
trainble = (W_1, W_2, W_3, W_4, b_1, b_2, b_3, b_4)
sgd = SGD(trainble, 3e-3, momentum=0.7)

In [78]:
def Loss(y_pred, y_actual, weights):
    y_actual_pred = (y_pred * y_actual).sum(axis=1)
    log_probs = -y_actual_pred + y_pred.exp().sum(axis=1).log()
    sample_weights = (y_actual * weights).sum(axis=1)
    return (log_probs * sample_weights).sum() / sample_weights.sum()

In [79]:
batch_size = 35
for epoch in range(100):
    avg_loss = []
    epoch_indx = np.random.permutation(len(X_train_scaled))
    for start in range(0, len(X_train_scaled), batch_size):
        sgd.zero_grad()
        
        batch_indx = epoch_indx[start:start + batch_size]
        X_batch = Tensor(X_train_scaled[batch_indx], 'cuda')
        y_batch = Tensor(y_train[batch_indx], 'cuda')
        
        y_pred = forward(X_batch, *trainble)

        loss = Loss(y_pred, y_batch, weights)
        avg_loss.append(loss.data.item())
        backward(loss)
        sgd.step()

    X_test_tensor = Tensor(X_test_scaled, 'cuda')
    y_test_tensor = Tensor(y_test, 'cuda')
    y_pred_test = forward(X_test_tensor, *trainble)
    test_loss = Loss(y_pred_test, y_test_tensor, weights)
    print(f'Epoch: {epoch + 1}, avg loss: {np.mean(avg_loss)}, test loss: {test_loss.data.item()}')

Epoch: 1, avg loss: 1.9181650971824473, test loss: 2.0160067081451416
Epoch: 2, avg loss: 1.8788045786998488, test loss: 2.075899600982666
Epoch: 3, avg loss: 1.846957310356877, test loss: 2.135394334793091
Epoch: 4, avg loss: 1.8247969915921038, test loss: 2.1678671836853027
Epoch: 5, avg loss: 1.7972289445725353, test loss: 2.1846015453338623
Epoch: 6, avg loss: 1.7616864930499683, test loss: 2.1917166709899902
Epoch: 7, avg loss: 1.7162922471761703, test loss: 2.1905601024627686
Epoch: 8, avg loss: 1.675670237703757, test loss: 2.181360960006714
Epoch: 9, avg loss: 1.6392681869593533, test loss: 2.1838085651397705
Epoch: 10, avg loss: 1.6005593463778496, test loss: 2.1940741539001465
Epoch: 11, avg loss: 1.57253843952309, test loss: 2.206501007080078
Epoch: 12, avg loss: 1.5437220612710172, test loss: 2.224297285079956
Epoch: 13, avg loss: 1.5241282385858623, test loss: 2.206484079360962
Epoch: 14, avg loss: 1.503774506124583, test loss: 2.211707592010498
Epoch: 15, avg loss: 1.4890